In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk(r'C:\Users\Admin\Desktop\Images\Retino\archive\dataset'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# import tensorflow as tf
# print(tf.__version__)

# !where python  # On Windows

!




In [3]:
# !pip install gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, gdown, cv2, shutil, math
from glob import glob
from sklearn.model_selection import train_test_split, KFold
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.layers import *  # Use tensorflow.keras instead of keras
from tensorflow.keras.models import Model, load_model, Sequential  # Use tensorflow.keras models
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Use tensorflow.keras preprocessing
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  # Use tensorflow.keras callbacks
# from tensorflow_addons.metrics import F1Score
from sklearn.metrics import confusion_matrix


In [4]:
# import tensorflow as tf
# from tensorflow.keras import backend as K

# def f1_score(y_true, y_pred):
#     # Clip predictions to prevent NaN errors due to floating-point precision
#     y_pred = K.clip(y_pred, K.epsilon(), 1 - K.epsilon())
    
#     # Calculate precision and recall
#     precision = K.sum(y_true * y_pred) / (K.sum(y_pred) + K.epsilon())
#     recall = K.sum(y_true * y_pred) / (K.sum(y_true) + K.epsilon())
    
#     # Calculate F1 score
#     f1 = 2 * (precision * recall) / (precision + recall + K.epsilon())
#     return f1

In [5]:
sdir = r'C:\Users\Admin\Desktop\Images\Retino\archive\dataset'


def make_dataframes(sdir):
    bad_images = []
    good_ext = ['jpg', 'jpeg', 'png', 'tiff']
    filepaths = []
    labels = []
    classes = sorted(os.listdir(sdir))
    for klass in classes:
        classpath = os.path.join(sdir, klass)
        flist = sorted(os.listdir(classpath))
        desc = f'{klass:23s}'
        for f in tqdm(flist, ncols=110, desc=desc, unit='file', colour='blue'):
            fpath = os.path.join(classpath, f)
            fl = f.lower()
            index = fl.rfind('.')
            ext = fl[index + 1:]
            if ext in good_ext:
                try:
                    img = cv2.imread(fpath)
                    shape = img.shape
                    filepaths.append(fpath)
                    labels.append(klass)
                except:
                    bad_images.append(fpath)
                    print('defective image file: ', fpath)
            else:
                bad_images.append(fpath)
    Fseries = pd.Series(filepaths, name='filepaths')
    Lseries = pd.Series(labels, name='labels')
    df = pd.concat([Fseries, Lseries], axis=1)

    train_df, dummy_df = train_test_split(df, train_size=.8, shuffle=True, random_state=123, stratify=df['labels'])
    valid_df, test_df = train_test_split(dummy_df, train_size=.5, shuffle=True, random_state=123,
                                         stratify=dummy_df['labels'])
    classes = sorted(train_df['labels'].unique())
    class_count = len(classes)
    sample_df = train_df.sample(n=50, replace=False)

    ht = 0
    wt = 0
    count = 0
    for i in range(len(sample_df)):
        fpath = sample_df['filepaths'].iloc[i]
        try:
            img = cv2.imread(fpath)
            h = img.shape[0]
            w = img.shape[1]
            wt += w
            ht += h
            count += 1
        except:
            pass
    have = int(ht / count)
    wave = int(wt / count)
    aspect_ratio = have / wave
    print('number of classes in processed dataset= ', class_count)
    counts = list(train_df['labels'].value_counts())
    print('the maximum files in any class in train_df is ', max(counts),
          ' the minimum files in any class in train_df is ', min(counts))
    print('train_df length: ', len(train_df), '  test_df length: ', len(test_df), '  valid_df length: ', len(valid_df))
    print('average image height= ', have, '  average image width= ', wave, ' aspect ratio h/w= ', aspect_ratio)
    return train_df, test_df, valid_df


train_df, test_df, valid_df = make_dataframes(sdir=sdir)

normal                 : 100%|█████████████████████████████████████████| 1074/1074 [00:01<00:00, 623.32file/s]


number of classes in processed dataset=  4
the maximum files in any class in train_df is  878  the minimum files in any class in train_df is  778
train_df length:  3321   test_df length:  416   valid_df length:  415
average image height=  457   average image width=  474  aspect ratio h/w=  0.9641350210970464


In [6]:
batch_size = 32
img_height, img_width = 300, 300
img_size = (img_height, img_width)
input_shape = (img_height, img_width, 3)

def create_data(batch_size, train_df, test_df, valid_df, img_size):
    train_gen = ImageDataGenerator(rescale=1/255.0, rotation_range=20, vertical_flip=True, shear_range=0.2, dtype=float)
    test_gen = ImageDataGenerator(rescale=1/255.0, dtype=float)
    
    train_set = train_gen.flow_from_dataframe(train_df, x_col='filepaths', y_col='labels', target_size=img_size, class_mode='categorical', batch_size=batch_size)
    test_set = test_gen.flow_from_dataframe(test_df, x_col='filepaths', y_col='labels', target_size=img_size, class_mode='categorical', batch_size=batch_size)
    val_set = test_gen.flow_from_dataframe(valid_df, x_col='filepaths', y_col='labels', target_size=img_size, class_mode='categorical', batch_size=batch_size)
    
    return train_set, test_set, val_set


train_set, test_set, val_set = create_data(batch_size, train_df, test_df, valid_df, img_size)

Found 3321 validated image filenames belonging to 4 classes.
Found 416 validated image filenames belonging to 4 classes.
Found 415 validated image filenames belonging to 4 classes.


In [7]:
label_dict = {v:k for k,v in train_set.class_indices.items()}
label_dict

{0: 'cataract', 1: 'diabetic_retinopathy', 2: 'glaucoma', 3: 'normal'}

In [8]:
def generate_model(pretrained_base,name=None):

    pretrained_base.trainable=False
    model = Sequential([
        pretrained_base,
        Flatten(),
        Dense(units=512, activation='relu'),
        Dense(units=256, activation='relu'),
        Dense(units=128, activation='relu'),
        Dense(units=64, activation='relu'),
        Dense(units=4, activation='softmax')
    ], name=name)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    
    return model

In [9]:
def generate_callbacks(name):
#     early_stop = EarlyStopping(monitor='val_loss', patience=3, min_delta=1e-4)
    checkpoint = ModelCheckpoint(filepath=f'best_{name}.ckpt', save_best_only=True, save_weights_only=True)
#     return [early_stop, checkpoint]
    return [checkpoint]

In [10]:
pretrained_params = {
    'weights':'imagenet',
    'include_top':False,
    'input_shape':input_shape
}


pretrained_models = {
    'ResNet50':tf.keras.applications.ResNet50(**pretrained_params),
    'EfficientNetB3':tf.keras.applications.EfficientNetB3(**pretrained_params),
    'MobileNet':tf.keras.applications.MobileNet(**pretrained_params),
    'DenseNet121':tf.keras.applications.DenseNet121(**pretrained_params)
}


fit_args = {
    'epochs':20,
    'steps_per_epoch':train_set.n//train_set.batch_size,
    'validation_data':val_set,
    'validation_steps':val_set.n//val_set.batch_size
}

In [11]:
# !pip install --upgrade tensorflow


In [12]:
model = Sequential(name='Baseline')

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.3))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(2048, activation='relu'))
model.add(Dense(1024, activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dense(4, activation='softmax'))


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

callbacks = generate_callbacks(name='baseline')

history = model.fit(train_set, callbacks=callbacks, **fit_args)

# best_model = model.load_weights('./best_baseline.ckpt')
model.save('./best_baseline.h5')

Epoch 1/20
103/103 [==============================] - 90s 868ms/step - loss: 1.3042 - categorical_accuracy: 0.4664 - val_loss: 0.9622 - val_categorical_accuracy: 0.5365
Epoch 2/20
103/103 [==============================] - 88s 850ms/step - loss: 0.8953 - categorical_accuracy: 0.5753 - val_loss: 0.7499 - val_categorical_accuracy: 0.6250
Epoch 3/20
103/103 [==============================] - 88s 855ms/step - loss: 0.7368 - categorical_accuracy: 0.6728 - val_loss: 0.6593 - val_categorical_accuracy: 0.7057
Epoch 4/20
103/103 [==============================] - 89s 866ms/step - loss: 0.6494 - categorical_accuracy: 0.7327 - val_loss: 0.7255 - val_categorical_accuracy: 0.7214
Epoch 5/20
103/103 [==============================] - 87s 841ms/step - loss: 0.5604 - categorical_accuracy: 0.7671 - val_loss: 0.7816 - val_categorical_accuracy: 0.7161
Epoch 6/20
103/103 [==============================] - 88s 851ms/step - loss: 0.5070 - categorical_accuracy: 0.7990 - val_loss: 0.5304 - val_categorical_acc

In [13]:
improved_baseline_model = Sequential(name='improved_baseline')

improved_baseline_model.add(Conv2D(32, (3,3), activation='relu', input_shape=input_shape))
improved_baseline_model.add(MaxPooling2D(2, 2))

improved_baseline_model.add(Conv2D(32, (3,3), activation='relu'))
improved_baseline_model.add(MaxPooling2D(2,2))
improved_baseline_model.add(Dropout(0.3))

improved_baseline_model.add(Conv2D(64, (3,3), activation='relu'))
improved_baseline_model.add(MaxPooling2D(2,2))

improved_baseline_model.add(Conv2D(128, (3,3), activation='relu'))
improved_baseline_model.add(MaxPooling2D(2,2))

improved_baseline_model.add(Flatten())

improved_baseline_model.add(Dense(2048, activation='relu'))
improved_baseline_model.add(Dense(1024, activation='relu'))
improved_baseline_model.add(Dense(128, activation='relu'))
improved_baseline_model.add(Dense(4, activation='softmax'))

improved_baseline_model.compile(optimizer='adam', 
                                loss='categorical_crossentropy', 
                                metrics=['categorical_accuracy'])

callbacks = generate_callbacks(name='improved_baseline')

history = improved_baseline_model.fit(train_set, callbacks=callbacks, **fit_args)

# best_model = improved_baseline_model.load_weights('./best_improved_baseline.ckpt')
improved_baseline_model.save('./best_improved_baseline.h5')

Epoch 1/20
103/103 [==============================] - 91s 876ms/step - loss: 1.2173 - categorical_accuracy: 0.4512 - val_loss: 0.8878 - val_categorical_accuracy: 0.5781
Epoch 2/20
103/103 [==============================] - 88s 852ms/step - loss: 0.8528 - categorical_accuracy: 0.6023 - val_loss: 0.7295 - val_categorical_accuracy: 0.6823
Epoch 3/20
103/103 [==============================] - 87s 844ms/step - loss: 0.6950 - categorical_accuracy: 0.7033 - val_loss: 0.9889 - val_categorical_accuracy: 0.6875
Epoch 4/20
103/103 [==============================] - 89s 864ms/step - loss: 0.5746 - categorical_accuracy: 0.7631 - val_loss: 0.5082 - val_categorical_accuracy: 0.7995
Epoch 5/20
103/103 [==============================] - 88s 851ms/step - loss: 0.5040 - categorical_accuracy: 0.7960 - val_loss: 0.6288 - val_categorical_accuracy: 0.7708
Epoch 6/20
103/103 [==============================] - 88s 852ms/step - loss: 0.4942 - categorical_accuracy: 0.8063 - val_loss: 0.5452 - val_categorical_acc

In [14]:
ResNet50_model = generate_model(pretrained_models['ResNet50'], name='ResNet50')
callbacks = generate_callbacks('ResNet50')
history = ResNet50_model.fit(train_set, callbacks=callbacks, **fit_args)

# best_model = ResNet50_model.load_weights('./best_ResNet50.ckpt')
ResNet50_model.save('./best_ResNet50.h5')

Epoch 1/20
103/103 [==============================] - 151s 1s/step - loss: 4.9704 - categorical_accuracy: 0.3171 - val_loss: 2.2391 - val_categorical_accuracy: 0.2344
Epoch 2/20
103/103 [==============================] - 150s 1s/step - loss: 1.5404 - categorical_accuracy: 0.3849 - val_loss: 1.3084 - val_categorical_accuracy: 0.4583
Epoch 3/20
103/103 [==============================] - 151s 1s/step - loss: 1.2640 - categorical_accuracy: 0.4588 - val_loss: 1.0485 - val_categorical_accuracy: 0.5625
Epoch 4/20
103/103 [==============================] - 150s 1s/step - loss: 1.1317 - categorical_accuracy: 0.5065 - val_loss: 0.9369 - val_categorical_accuracy: 0.6224
Epoch 5/20
103/103 [==============================] - 149s 1s/step - loss: 1.0514 - categorical_accuracy: 0.5336 - val_loss: 1.3208 - val_categorical_accuracy: 0.3724
Epoch 6/20
103/103 [==============================] - 155s 2s/step - loss: 0.9956 - categorical_accuracy: 0.5509 - val_loss: 1.1398 - val_categorical_accuracy: 0.507

In [15]:
EfficientNetB3_model = generate_model(pretrained_models['EfficientNetB3'], name='EfficientNetB3')
callbacks = generate_callbacks('EfficientNetB3')
history = EfficientNetB3_model.fit(train_set, callbacks=callbacks, **fit_args)

# Ensure best weights are loaded before saving
# EfficientNetB3_model.load_weights('./best_EfficientNetB3.ckpt')

# Save only the weights (fixes the serialization issue)
EfficientNetB3_model.save_weights('./best_EfficientNetB3_weights.h5')


Epoch 1/20
103/103 [==============================] - 135s 1s/step - loss: 8.1120 - categorical_accuracy: 0.2408 - val_loss: 3.5703 - val_categorical_accuracy: 0.2656
Epoch 2/20
103/103 [==============================] - 136s 1s/step - loss: 1.7975 - categorical_accuracy: 0.2551 - val_loss: 1.6246 - val_categorical_accuracy: 0.2578
Epoch 3/20
103/103 [==============================] - 131s 1s/step - loss: 1.6321 - categorical_accuracy: 0.2578 - val_loss: 1.4183 - val_categorical_accuracy: 0.2839
Epoch 4/20
103/103 [==============================] - 126s 1s/step - loss: 1.5218 - categorical_accuracy: 0.2542 - val_loss: 1.5979 - val_categorical_accuracy: 0.2318
Epoch 5/20
103/103 [==============================] - 126s 1s/step - loss: 1.4432 - categorical_accuracy: 0.2600 - val_loss: 1.4416 - val_categorical_accuracy: 0.2760
Epoch 6/20
103/103 [==============================] - 128s 1s/step - loss: 1.3999 - categorical_accuracy: 0.2770 - val_loss: 1.4127 - val_categorical_accuracy: 0.234

In [16]:
MobileNet_model = generate_model(pretrained_models['MobileNet'], name='MobileNet')
callbacks = generate_callbacks('MobileNet')
history = MobileNet_model.fit(train_set, callbacks=callbacks, **fit_args)

# best_model = MobileNet_model.load_weights('./best_MobileNet.ckpt')
MobileNet_model.save('./best_MobileNet.h5')

Epoch 1/20
103/103 [==============================] - 76s 722ms/step - loss: 2.5583 - categorical_accuracy: 0.7543 - val_loss: 1.5546 - val_categorical_accuracy: 0.7214
Epoch 2/20
103/103 [==============================] - 74s 716ms/step - loss: 0.6958 - categorical_accuracy: 0.8379 - val_loss: 0.6908 - val_categorical_accuracy: 0.8307
Epoch 3/20
103/103 [==============================] - 76s 741ms/step - loss: 0.2571 - categorical_accuracy: 0.9082 - val_loss: 0.3883 - val_categorical_accuracy: 0.8828
Epoch 4/20
103/103 [==============================] - 75s 725ms/step - loss: 0.2426 - categorical_accuracy: 0.9167 - val_loss: 0.4631 - val_categorical_accuracy: 0.8620
Epoch 5/20
103/103 [==============================] - 74s 722ms/step - loss: 0.1908 - categorical_accuracy: 0.9276 - val_loss: 0.3569 - val_categorical_accuracy: 0.8698
Epoch 6/20
103/103 [==============================] - 74s 715ms/step - loss: 0.1455 - categorical_accuracy: 0.9453 - val_loss: 0.3778 - val_categorical_acc

In [17]:
DenseNet121_model = generate_model(pretrained_models['DenseNet121'], name='DenseNet121')
callbacks = generate_callbacks('DenseNet121')
history = DenseNet121_model.fit(train_set, callbacks=callbacks, **fit_args)

# best_model = DenseNet121_model.load_weights('./best_DenseNet121.ckpt')
DenseNet121_model.save('./best_DenseNet121.h5')

Epoch 1/20
103/103 [==============================] - 131s 1s/step - loss: 2.4002 - categorical_accuracy: 0.7188 - val_loss: 0.4179 - val_categorical_accuracy: 0.8724
Epoch 2/20
103/103 [==============================] - 123s 1s/step - loss: 0.3454 - categorical_accuracy: 0.8772 - val_loss: 0.4906 - val_categorical_accuracy: 0.8594
Epoch 3/20
103/103 [==============================] - 119s 1s/step - loss: 0.3565 - categorical_accuracy: 0.8805 - val_loss: 0.5883 - val_categorical_accuracy: 0.8359
Epoch 4/20
103/103 [==============================] - 124s 1s/step - loss: 0.2664 - categorical_accuracy: 0.8960 - val_loss: 0.4227 - val_categorical_accuracy: 0.8516
Epoch 5/20
103/103 [==============================] - 127s 1s/step - loss: 0.2143 - categorical_accuracy: 0.9188 - val_loss: 0.4038 - val_categorical_accuracy: 0.8646
Epoch 6/20
103/103 [==============================] - 121s 1s/step - loss: 0.2190 - categorical_accuracy: 0.9158 - val_loss: 0.3582 - val_categorical_accuracy: 0.888

In [ ]:
resnet = load_model('./best_ResNet50.h5')
efficientNet = generate_model(pretrained_models['EfficientNetB3'], name='EfficientNetB3')  # Rebuild model
efficientNet.load_weights('./best_EfficientNetB3_weights.h5')  # Load weights correctly
mobilenet = load_model('./best_MobileNet.h5')
densenet = load_model('./best_DenseNet121.h5')
baseline = load_model('./best_baseline.h5')
improved_baseline = load_model('./best_improved_baseline.h5')

inputs = Input(shape=input_shape)
resnet_preds = resnet(inputs)
efficientNet_preds = efficientNet(inputs)
mobilenet_preds = mobilenet(inputs)
densenet_preds = densenet(inputs)
baseline_preds = baseline(inputs)
improved_baseline_preds = improved_baseline(inputs)

outputs = Average()([
    resnet_preds,
    efficientNet_preds,
    mobilenet_preds,
    densenet_preds,
    baseline_preds,
    improved_baseline_preds])

ensemble = Model(inputs=inputs, outputs=outputs, name='Ensemble')
ensemble.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
# ensemble.trainable = False

callbacks = generate_callbacks('Ensemble')
history = ensemble.fit(train_set, callbacks=callbacks, **fit_args)

Epoch 1/20


In [ ]:
train_loss, train_acc, train_f1, val_loss, val_acc, val_f1, \
test_loss, test_acc, test_f1 = [], [], [], [], [], [], [], [], []

models = {
    'baseline': baseline,
    'improved_baseline': improved_baseline,
    'vgg': resnet,
    'xception': efficientNet,
    'densenet': densenet,
    'mobilenet': mobilenet,
    'ensemble': ensemble
}

for name, model in tqdm(models.items()):
    print(f'Evaluating {name}')
    train_l, train_a = model.evaluate(train_set)
    val_l, val_a = model.evaluate(val_set)
    test_l, test_a = model.evaluate(test_set)
    train_loss.append(train_l)
    train_acc.append(train_a)
    # train_f1.append(np.mean(train_f))
    val_loss.append(val_l)
    val_acc.append(val_a)
    # val_f1.append(np.mean(val_f))
    test_loss.append(test_l)
    test_acc.append(test_a)
    # test_f1.append(np.mean(test_f)


performance = {
    'Train Loss': train_loss,
    'Validation Loss': val_loss,
    'Test Loss': test_loss,
    'Train Accuracy': train_acc,
    'Validation Accuracy': val_acc,
    'Test Accuracy': test_acc
    # 'Train F1': train_f1,
    # 'Validation F1': val_f1,
    # 'Test F1': test_f1
}

performance_df = pd.DataFrame(performance, index=models.keys())